# 04. 병합 (카드 + 기상 + 달력)

**먼저 실행할 것**: 01, 03 노트북

**만드는 것**: `data/processed/analysis_visitor.csv`, `analysis_time.csv`

## 1. 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
RAW_CARD = BASE / "data/raw/card"
EXT = BASE / "data/external"
OUT = BASE / "data/processed"
DOCS = BASE / "docs"
OUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

In [ ]:
holidays = pd.to_datetime(["2025-08-15", "2025-10-03", "2025-10-05", "2025-10-06",
                           "2025-10-07", "2025-10-08", "2025-10-09", "2025-12-25"])

cal = pd.DataFrame({"date": pd.date_range("2025-07-01", "2025-12-31")})
cal["dow"] = cal["date"].dt.dayofweek                 # 0=월 … 6=일
cal["weekend"] = (cal["dow"] >= 5).astype(int)
cal["holiday"] = cal["date"].isin(holidays).astype(int)
cal["offday"] = ((cal["weekend"] == 1) | (cal["holiday"] == 1)).astype(int)
cal["chuseok"] = cal["date"].between("2025-10-03", "2025-10-09").astype(int)
cal["month"] = cal["date"].dt.month
cal["week"] = cal["date"].dt.isocalendar().week.astype(int)
cal["summer"] = (cal["month"] <= 9).astype(int)       # 폭염 분석 기간 (7~9월)
print(cal.shape)                                      # (184, 9) 예상

## 3. 읽기

CSV는 날짜 형식을 기억하지 못하므로 `parse_dates`로 꼭 변환.

In [ ]:
pv = pd.read_csv(OUT / "panel_visitor.csv", parse_dates=["date"])
pt = pd.read_csv(OUT / "panel_time.csv", parse_dates=["date"])
weather_d = pd.read_csv(OUT / "weather_daily.csv", parse_dates=["date"])
weather_t = pd.read_csv(OUT / "weather_timegb.csv", parse_dates=["date"])

## 4. 병합

In [ ]:
analysis_visitor = (pv.merge(cal, on="date", how="left")
                      .merge(weather_d, on=["region", "date"], how="left"))

analysis_time = (pt.merge(cal, on="date", how="left")
                   .merge(weather_d, on=["region", "date"], how="left")
                   .merge(weather_t, on=["region", "date", "time_gb"], how="left"))

analysis_visitor.to_csv(OUT / "analysis_visitor.csv", index=False, encoding="utf-8-sig")
analysis_time.to_csv(OUT / "analysis_time.csv", index=False, encoding="utf-8-sig")

## 5. 점검

결측 0, 지역별 폭염일 24가 나오면 성공.

In [ ]:
for name, df in [("visitor", analysis_visitor), ("time", analysis_time)]:
    print(f"[{name}] {df.shape} | 결측 {df.drop(columns=['tmax_lag1']).isna().sum().sum()}")
    print(df.columns.tolist())

print(analysis_visitor.groupby(["region", "heat"])["date"].nunique().unstack())